### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
> 4. Submit your homework as html file just like our exercise did.
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [1]:
from collections import Counter
import json
from pathlib import Path

# 说明：Path.cwd() 会返回当前 notebook 运行时所在的工作目录。
# 本次作业的数据文件与 ipynb 文件放在同一目录下，因此可以直接基于当前目录拼接文件路径。
DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "enwiki-train.json"
TEST_PATH = DATA_DIR / "enwiki-test.json"


def load_jsonl(file_path: Path):
    """读取 JSON Lines 文件，并将每一行解析为一个字典。

    参数:
        file_path: JSONL 文件路径。当前数据集中每一行都是一条 Wikipedia 文档记录。

    返回:
        records: 列表，其中每个元素都是包含 title、label、text 等字段的字典。
    """
    records = []

    # 使用 utf-8 编码打开文件，逐行读取可以避免一次性把整个大文件作为单个字符串载入内存。
    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            # strip() 用于去掉每行末尾的换行符；空行直接跳过，增强代码健壮性。
            line = line.strip()
            if not line:
                continue

            # json.loads() 负责把 JSON 格式的字符串解析成 Python 字典。
            records.append(json.loads(line))

    return records


def count_documents_by_label(records):
    """统计每个类别中包含多少篇文档。"""

    # Counter 是标准库中专门用于计数的容器，这里统计每条记录的 label 字段出现次数。
    return Counter(record["label"] for record in records)


# 读取训练集和测试集，为后续 Task1 其他小题以及后面的建模任务做准备。
train_records = load_jsonl(TRAIN_PATH)
test_records = load_jsonl(TEST_PATH)

# 分别统计训练集与测试集中每个类别对应的文档数量。
train_label_counts = count_documents_by_label(train_records)
test_label_counts = count_documents_by_label(test_records)

# sorted() 按类别名称排序，方便 notebook 输出稳定、清晰，也便于和测试集结果逐行比较。
print("Training dataset - number of documents in each class:")
for label, count in sorted(train_label_counts.items()):
    print(f"{label}: {count}")

print("\nTesting dataset - number of documents in each class:")
for label, count in sorted(test_label_counts.items()):
    print(f"{label}: {count}")


Training dataset - number of documents in each class:
Actor: 79
Animal: 82
Artist: 457
Book: 858
Disease: 202
Film: 2752
Food: 121
Politician: 3441
Software: 239
Writer: 769

Testing dataset - number of documents in each class:
Actor: 1
Animal: 11
Artist: 63
Book: 117
Disease: 18
Film: 296
Food: 16
Politician: 383
Software: 27
Writer: 68


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [2]:
import spacy
from collections import defaultdict

# 这里使用 spacy.blank("en") 构造一个轻量级英文处理管道。
# 与需要额外下载语言模型的方案相比，这种写法更适合作业环境，通常不依赖网络。
nlp = spacy.blank("en")

# sentencizer 是 spaCy 提供的基于规则的分句组件。
# 它会依据标点等规则切分句子，适合本题统计平均句子数的需求。
if "sentencizer" not in nlp.pipe_names:
    nlp.add_pipe("sentencizer")


def average_sentence_count_by_label(records, nlp, batch_size: int = 64):
    """计算每个类别中文档的平均句子数。

    参数:
        records: 由多个文档字典组成的列表，每个字典至少包含 label 和 text 字段。
        nlp: spaCy 语言处理管道，这里主要用于句子切分。
        batch_size: nlp.pipe() 的批处理大小。适当分批可以提高处理长文本时的效率。

    返回:
        average_counts: 字典，键为类别名，值为该类别文档的平均句子数。
    """
    # sentence_totals 用于累计某个类别下所有文档的句子总数。
    sentence_totals = defaultdict(int)

    # document_totals 用于累计某个类别下的文档总数，后面计算平均值时要作为分母。
    document_totals = defaultdict(int)

    # nlp.pipe() 可以批量处理文本，比对每条文本单独调用 nlp(text) 更高效。
    # 这里用 zip(records, ...) 保持“原始记录”和“处理后的 doc 对象”一一对应。
    for record, doc in zip(records, nlp.pipe((record["text"] for record in records), batch_size=batch_size)):
        label = record["label"]

        # doc.sents 会返回当前文档分好的句子。
        # 这里额外用 strip() 过滤掉空白句子，保证统计更稳健。
        sentence_count = sum(1 for sent in doc.sents if sent.text.strip())

        sentence_totals[label] += sentence_count
        document_totals[label] += 1

    # 对每个类别计算“句子总数 / 文档总数”，得到平均句子数。
    average_counts = {
        label: sentence_totals[label] / document_totals[label]
        for label in sorted(document_totals)
    }
    return average_counts


# 分别统计训练集与测试集中，每个类别对应的平均句子数。
train_avg_sentence_counts = average_sentence_count_by_label(train_records, nlp)
test_avg_sentence_counts = average_sentence_count_by_label(test_records, nlp)

print("Training dataset - average number of sentences in each class:")
for label, avg_count in train_avg_sentence_counts.items():
    print(f"{label}: {avg_count:.2f}")

print("\nTesting dataset - average number of sentences in each class:")
for label, avg_count in test_avg_sentence_counts.items():
    print(f"{label}: {avg_count:.2f}")


Training dataset - average number of sentences in each class:
Actor: 69.33
Animal: 67.06
Artist: 185.94
Book: 205.08
Disease: 347.86
Film: 177.88
Food: 153.44
Politician: 222.86
Software: 201.13
Writer: 216.11

Testing dataset - average number of sentences in each class:
Actor: 177.00
Animal: 62.82
Artist: 174.97
Book: 197.96
Disease: 325.89
Film: 173.68
Food: 165.12
Politician: 233.03
Software: 204.00
Writer: 220.60


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [ ]:
# 这里继续复用上一个代码块中已经构建好的 nlp、train_records 和 test_records。
# 注意：本小题统计的是“原始文本经过 spaCy 分词后的 token 数量”，
# 只排除纯空白 token，不额外去掉标点，因为题目此处尚未要求先做文本清洗。

def average_token_count_by_label(records, nlp, batch_size: int = 64):
    """计算每个类别中文档的平均 token 数。

    参数:
        records: 文档字典列表，每条记录至少包含 label 和 text 字段。
        nlp: spaCy 语言处理管道。这里主要复用它的英文 tokenizer。
        batch_size: nlp.pipe() 的批处理大小，用于提升批量处理效率。

    返回:
        average_counts: 字典，键为类别名，值为该类别文档的平均 token 数。
    """
    # token_totals 用于累计某个类别下所有文档的 token 总数。
    token_totals = defaultdict(int)

    # document_totals 用于累计某个类别下的文档数量，后面会作为平均值的分母。
    document_totals = defaultdict(int)

    # 继续使用 nlp.pipe() 批量处理文本，避免逐条调用 tokenizer 带来的额外开销。
    for record, doc in zip(records, nlp.pipe((record["text"] for record in records), batch_size=batch_size)):
        label = record["label"]

        # token.is_space 用来判断当前 token 是否只是空白符。
        # 这里保留普通单词、数字和标点 token，只过滤没有实际内容的空白 token。
        token_count = sum(1 for token in doc if not token.is_space)

        token_totals[label] += token_count
        document_totals[label] += 1

    # 计算每个类别的平均 token 数。
    average_counts = {
        label: token_totals[label] / document_totals[label]
        for label in sorted(document_totals)
    }
    return average_counts


# 分别统计训练集和测试集中各类别文档的平均 token 数。
train_avg_token_counts = average_token_count_by_label(train_records, nlp)
test_avg_token_counts = average_token_count_by_label(test_records, nlp)

print("Training dataset - average number of tokens in each class:")
for label, avg_count in train_avg_token_counts.items():
    print(f"{label}: {avg_count:.2f}")

print("\nTesting dataset - average number of tokens in each class:")
for label, avg_count in test_avg_token_counts.items():
    print(f"{label}: {avg_count:.2f}")


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [ ]:
import re

# 正则表达式 [a-z0-9]+ 表示：只保留连续的英文字母和数字。
# 这样可以移除标点符号、特殊字符以及其他不需要的符号。
TOKEN_PATTERN = re.compile(r"[a-z0-9]+")


def preprocess_sentence(sentence_text: str):
    """清洗单个句子，只保留英文单词和数字，并统一转换为小写。

    参数:
        sentence_text: 原始句子字符串。

    返回:
        cleaned_tokens: 清洗后的 token 列表。
    """
    # lower() 先把所有字符转成小写，满足题目中“make all words as lower cases”的要求。
    lowered_text = sentence_text.lower()

    # findall() 会提取所有满足正则模式的片段。
    # 最终保留下来的只有英文字母和数字组成的 token。
    cleaned_tokens = TOKEN_PATTERN.findall(lowered_text)
    return cleaned_tokens


def preprocess_document(text: str, nlp):
    """对整篇文档进行预处理，并按句子保留清洗后的结果。

    返回:
        processed_sentences: 二维列表。外层列表表示句子，内层列表表示该句子中的 token。
    """
    processed_sentences = []

    # 先用前面已经创建好的 sentencizer 对文档进行分句。
    doc = nlp(text)

    for sent in doc.sents:
        cleaned_tokens = preprocess_sentence(sent.text)

        # 如果某个句子清洗后变成空列表，说明它可能只包含符号或空白，不保留。
        if cleaned_tokens:
            processed_sentences.append(cleaned_tokens)

    return processed_sentences


def attach_processed_text(records, nlp):
    """为每条文档记录补充可复用的预处理结果。

    新增字段说明:
        processed_sentences: 按句子组织的清洗后 token。
        processed_tokens: 将所有句子的 token 展平后的列表，后面语言模型任务可以直接复用。
        processed_text: 把展平后的 token 重新拼接成一个字符串，便于展示或后续建模使用。
    """
    for record in records:
        processed_sentences = preprocess_document(record["text"], nlp)

        # 展平二维列表，得到整篇文档的 token 序列。
        processed_tokens = [token for sentence in processed_sentences for token in sentence]

        record["processed_sentences"] = processed_sentences
        record["processed_tokens"] = processed_tokens
        record["processed_text"] = " ".join(processed_tokens)


def get_first_record_of_each_class(records):
    """按照数据集原始顺序，找到每个类别第一次出现的文章。"""
    first_records = {}

    for record in records:
        label = record["label"]
        if label not in first_records:
            first_records[label] = record

    return first_records


# 对训练集和测试集分别补充预处理结果。
# 这些字段会在后续 Task2 中继续使用，因此这里提前保存到原始记录中。
attach_processed_text(train_records, nlp)
attach_processed_text(test_records, nlp)

train_first_records = get_first_record_of_each_class(train_records)
test_first_records = get_first_record_of_each_class(test_records)

print("Training dataset - first article in each class and its first 40 processed words:")
for label in sorted(train_first_records):
    record = train_first_records[label]
    preview_words = " ".join(record["processed_tokens"][:40])
    print(f"{label} | title: {record['title']}")
    print(f"first 40 words: {preview_words}")
    print()

print("Testing dataset - first article in each class and its first 40 processed words:")
for label in sorted(test_first_records):
    record = test_first_records[label]
    preview_words = " ".join(record["processed_tokens"][:40])
    print(f"{label} | title: {record['title']}")
    print(f"first 40 words: {preview_words}")
    print()


### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [ ]:
import math
import random

# ==============================
# Step 1. 准备语言模型训练所需的语料
# ==============================
# 在 Task1 的第 4 小题里，我们已经把每篇文档做了基础清洗，
# 并把结果保存在了 processed_sentences 字段中。
#
# processed_sentences 的数据结构长这样：
# [
#     ["this", "is", "sentence", "one"],
#     ["this", "is", "sentence", "two"]
# ]
# 也就是说：
# - 外层列表表示“一篇文档中的多个句子”
# - 内层列表表示“某一个句子中的多个 token”
#
# 如果你前面已经顺序运行过 notebook，这些字段应该已经存在。
# 这里加一个保护判断：如果它们不存在，就自动调用前面写好的函数补上。
if "processed_sentences" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_sentences" not in test_records[0]:
    attach_processed_text(test_records, nlp)


def collect_all_sentences(records):
    """把数据集中的所有句子收集到一个列表里。

    题目要求“collect all sentences in training dataset”，
    所以这里不是按文档训练语言模型，而是把训练集中的所有句子合并成一个总语料。

    参数:
        records: 文档记录列表，每条记录应包含 processed_sentences 字段。

    返回:
        all_sentences: 一个大列表，其中每个元素都是“一个句子的 token 列表”。
    """
    all_sentences = []

    for record in records:
        # extend() 会把另一个列表里的元素逐个加入当前列表。
        # 这里的意思是：把“这一篇文档里的所有句子”都加入总语料中。
        all_sentences.extend(record["processed_sentences"])

    return all_sentences


train_sentences_raw = collect_all_sentences(train_records)
test_sentences_raw = collect_all_sentences(test_records)

# ==============================
# Step 2. 构建词表（Vocabulary）并处理低频词
# ==============================
# 为什么要先建词表？
# 因为语言模型后面所有概率计算，都是围绕“词表中的词”展开的。
#
# 为什么要把低频词换成 <UNK>？
# 因为出现次数太少的词很难学到稳定统计规律。
# 如果我们把这些稀有词全部保留，词表会变得非常大，而且非常稀疏。
# 所以通常会把低频词统一映射成一个特殊符号 <UNK>。
#
# 这里我们严格按照题目提示，使用 cutoff = 10。
UNK_TOKEN = "<UNK>"
START_TOKEN = "<s>"
END_TOKEN = "</s>"
MIN_FREQUENCY = 10
ADDITIVE_ALPHA = 1.0


def build_vocabulary(sentences, min_frequency: int = 10):
    """根据训练句子构建词表。

    参数:
        sentences: 句子列表，每个句子又是一个 token 列表。
        min_frequency: 最小保留词频。低于这个次数的词不会单独留在词表中。

    返回:
        vocabulary: 最终词表（set 类型）。
        token_counter: 每个词在训练集中出现次数的统计结果。
    """
    # Counter 是 Python 标准库里的计数器。
    # 例如 Counter(["a", "b", "a"]) 会得到 {"a": 2, "b": 1}。
    token_counter = Counter()

    for sentence in sentences:
        token_counter.update(sentence)

    # 只保留训练集中出现次数 >= min_frequency 的词。
    vocabulary = {
        token
        for token, count in token_counter.items()
        if count >= min_frequency
    }

    # 把几个特殊符号也加入词表。
    # <UNK> 表示未知词或低频词；<s> 和 </s> 用于标记句子边界。
    vocabulary.update({UNK_TOKEN, START_TOKEN, END_TOKEN})

    return vocabulary, token_counter


def replace_rare_and_oov_tokens(sentences, vocabulary):
    """把不在词表中的词统一替换为 <UNK>。

    这个函数会同时用于训练集和测试集：
    - 训练集里，低频词会变成 <UNK>
    - 测试集里，训练词表中没见过的新词也会变成 <UNK>
    """
    normalized_sentences = []

    for sentence in sentences:
        normalized_sentence = [
            token if token in vocabulary else UNK_TOKEN
            for token in sentence
        ]
        normalized_sentences.append(normalized_sentence)

    return normalized_sentences


vocabulary, train_token_counter = build_vocabulary(
    train_sentences_raw,
    min_frequency=MIN_FREQUENCY,
)

# train_sentences 和 test_sentences 将会是后面所有语言模型任务真正使用的语料。
# 也就是说，从这里开始，我们都默认“词表已经固定、低频词已经替换成 <UNK>”。
train_sentences = replace_rare_and_oov_tokens(train_sentences_raw, vocabulary)
test_sentences = replace_rare_and_oov_tokens(test_sentences_raw, vocabulary)

# 词表既保留 set 版本，也保留 list 版本。
# - set：适合快速判断某个词是否存在
# - list：适合后面按固定顺序遍历所有候选词
vocabulary_list = sorted(vocabulary)
vocabulary_size = len(vocabulary_list)

# ==============================
# Step 3. 自己实现 Additive Smoothing 的 N-Gram 语言模型
# ==============================
# 先说最核心的思想：
#
# 1. Unigram model
#    只看“某个词自己出现得多不多”。
#    例如：P("book")
#
# 2. Bigram model
#    看“前一个词出现时，当前词出现的概率”。
#    例如：P("book" | "this")
#
# 3. Trigram model
#    看“前两个词出现时，当前词出现的概率”。
#    例如：P("book" | "this", "is")
#
# 如果某个搭配在训练集中从来没出现过，普通统计方法会给它 0 概率。
# 一旦句子中某一步概率为 0，整句概率都会被乘成 0。
# 为了解决这个问题，我们要做平滑（smoothing）。
#
# 加性平滑公式如下：
# P(w | context) = (count(context, w) + alpha) / (count(context) + alpha * |V|)
#
# 其中：
# - count(context, w)：上下文 context 后面接 w 的次数
# - count(context)：这个上下文总共出现了多少次
# - |V|：词表大小
# - alpha：平滑系数。alpha = 1.0 时也常叫 Laplace smoothing


class AdditiveNGramLanguageModel:
    """使用加性平滑的 N-Gram 语言模型。

    这个类会在后面的多个小题中复用：
    - 当前小题：训练模型
    - 下一小题：计算 perplexity
    - 再下一小题：生成句子
    """

    def __init__(self, n: int, vocabulary, alpha: float = 1.0):
        """初始化模型。

        参数:
            n: n-gram 中的 n。1=unigram，2=bigram，3=trigram。
            vocabulary: 词表。
            alpha: 加性平滑系数。
        """
        self.n = n
        self.alpha = alpha
        self.vocabulary = set(vocabulary)
        self.vocabulary_list = sorted(vocabulary)
        self.vocabulary_size = len(self.vocabulary_list)

        # ngram_counts 用来统计完整 n-gram 的出现次数。
        # 例如在 bigram 中，它会统计 ('new', 'york') 出现了几次。
        self.ngram_counts = Counter()

        # context_counts 用来统计“上下文”的出现次数。
        # 例如：
        # - unigram 的上下文是空的
        # - bigram 的上下文是前 1 个词
        # - trigram 的上下文是前 2 个词
        self.context_counts = Counter()

    def pad_sentence(self, sentence):
        """给句子补上开始和结束标记。

        为什么要这样做？
        因为模型不仅要学“词和词之间怎么接”，
        还要学“句子怎么开始、句子怎么结束”。

        例如 trigram 需要两个前文词作为上下文，
        所以句子开头需要补两个 <s>。
        """
        if self.n == 1:
            return sentence + [END_TOKEN]

        start_tokens = [START_TOKEN] * (self.n - 1)
        return start_tokens + sentence + [END_TOKEN]

    def fit(self, sentences):
        """根据训练句子统计 n-gram 频数。"""
        for sentence in sentences:
            padded_sentence = self.pad_sentence(sentence)

            # 用一个长度为 n 的滑动窗口从左到右扫描句子。
            for i in range(len(padded_sentence) - self.n + 1):
                ngram = tuple(padded_sentence[i : i + self.n])

                # context 就是去掉最后一个词之后剩下的部分。
                # 例如 trigram ('i', 'love', 'nlp') 的 context 是 ('i', 'love')。
                context = ngram[:-1]

                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

    def get_probability(self, token, context=None):
        """计算条件概率 P(token | context)。"""
        if context is None:
            context = ()

        # 对于 bigram / trigram，只保留最后 n-1 个词作为真正上下文。
        context = tuple(context[-(self.n - 1) :]) if self.n > 1 else ()

        # 如果 token 不在训练词表中，就把它当作 <UNK>。
        if token not in self.vocabulary:
            token = UNK_TOKEN

        ngram = context + (token,)
        ngram_count = self.ngram_counts[ngram]
        context_count = self.context_counts[context]

        numerator = ngram_count + self.alpha
        denominator = context_count + self.alpha * self.vocabulary_size
        return numerator / denominator

    def sentence_log_probability(self, sentence):
        """计算一个句子的对数概率。"""
        padded_sentence = self.pad_sentence(sentence)
        log_probability = 0.0

        for i in range(len(padded_sentence) - self.n + 1):
            ngram = tuple(padded_sentence[i : i + self.n])
            context = ngram[:-1]
            token = ngram[-1]
            probability = self.get_probability(token, context)

            # 概率连乘会非常小，所以先取 log，再把乘法变成加法，更稳定。
            log_probability += math.log(probability)

        return log_probability

    def perplexity(self, sentences):
        """计算一组句子的困惑度。

        直观理解：
        困惑度越小，说明模型越擅长预测这些句子。
        这个函数会在 Task2 的第 2 小题直接复用。
        """
        total_log_probability = 0.0
        total_predicted_tokens = 0

        for sentence in sentences:
            padded_sentence = self.pad_sentence(sentence)
            total_log_probability += self.sentence_log_probability(sentence)
            total_predicted_tokens += len(padded_sentence) - self.n + 1

        return math.exp(-total_log_probability / total_predicted_tokens)

    def generate_sentence(self, max_length: int = 20, random_seed=None):
        """用当前模型随机生成一句话。

        这个函数会在 Task2 的第 3 小题中直接复用。
        """
        if random_seed is not None:
            random.seed(random_seed)

        generated_tokens = []
        context = [START_TOKEN] * (self.n - 1) if self.n > 1 else []

        # 生成时不允许把 <s> 当作普通输出词。
        candidate_tokens = [token for token in self.vocabulary_list if token != START_TOKEN]

        for _ in range(max_length):
            probabilities = [self.get_probability(token, context) for token in candidate_tokens]
            next_token = random.choices(candidate_tokens, weights=probabilities, k=1)[0]

            if next_token == END_TOKEN:
                break

            generated_tokens.append(next_token)

            if self.n > 1:
                context = (context + [next_token])[-(self.n - 1) :]

        return generated_tokens


# ==============================
# Step 4. 训练 unigram / bigram / trigram 模型
# ==============================
unigram_model = AdditiveNGramLanguageModel(n=1, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)
bigram_model = AdditiveNGramLanguageModel(n=2, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)
trigram_model = AdditiveNGramLanguageModel(n=3, vocabulary=vocabulary_list, alpha=ADDITIVE_ALPHA)

unigram_model.fit(train_sentences)
bigram_model.fit(train_sentences)
trigram_model.fit(train_sentences)

# 用一个字典统一保存 3 个模型，后面调用会更方便。
language_models = {
    "unigram": unigram_model,
    "bigram": bigram_model,
    "trigram": trigram_model,
}

# 这里打印一些摘要信息，帮助你检查训练流程是否正常。
print("Language model preprocessing and training are ready.")
print(f"Number of training sentences: {len(train_sentences)}")
print(f"Number of testing sentences: {len(test_sentences)}")
print(f"Raw training vocabulary size: {len(train_token_counter)}")
print(f"Final vocabulary size after cutoff={MIN_FREQUENCY}: {vocabulary_size}")
print(f"Additive smoothing alpha: {ADDITIVE_ALPHA}")


> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [ ]:
# 这一小题的目标是：
# 1. 用上一题已经训练好的 unigram / bigram / trigram 模型
# 2. 在测试集句子上计算 perplexity（困惑度）
# 3. 打印结果，并给出一个简要解释
#
# 这里直接复用上一题中已经准备好的：
# - test_sentences
# - unigram_model
# - bigram_model
# - trigram_model
# - language_models

perplexity_results = {}

# 依次计算 3 个模型在测试集上的困惑度。
# 注意：perplexity 越小，通常表示模型越擅长预测这批测试句子。
for model_name, model in language_models.items():
    perplexity_results[model_name] = model.perplexity(test_sentences)

print("Perplexity on the testing dataset:")
for model_name, perplexity_value in perplexity_results.items():
    print(f"{model_name}: {perplexity_value:.4f}")


# ==============================
# 自动生成一段简短解释
# ==============================
# 这里不只打印数字，还顺手给出一个基础分析，方便直接写进作业。
# 这段解释是基于常见 n-gram 语言模型规律写的：
# - unigram 不看上下文，所以通常最弱
# - bigram 看 1 个词的上下文，通常会更好
# - trigram 看 2 个词的上下文，若训练数据足够，通常还能继续提升
#
# 但也不是上下文越长就一定越强。
# 如果数据太稀疏，trigram 也可能因为许多三元组合太少而效果不稳定。

sorted_results = sorted(perplexity_results.items(), key=lambda item: item[1])
best_model_name, best_perplexity = sorted_results[0]
worst_model_name, worst_perplexity = sorted_results[-1]

print("\nExplanation:")
print(
    f"The best model on the testing dataset is {best_model_name}, because it has the lowest perplexity ({best_perplexity:.4f})."
)
print(
    f"The weakest model among these three is {worst_model_name}, because its perplexity is the highest ({worst_perplexity:.4f})."
)
print(
    "In general, a lower perplexity means the model is less surprised by the testing sentences and predicts the next word more accurately."
)
print(
    "Usually, unigram performs worst because it ignores word order and context. Bigram and trigram can model local context, so they often achieve lower perplexity."
)
print(
    "If trigram is better than bigram, it suggests that using a longer context helps prediction. If trigram is not better, the reason is often data sparsity even after smoothing."
)


> 3) Use each built model to generate five sentences and explain these generated patterns.


In [ ]:
# 这一小题要求：
# - 分别用 unigram / bigram / trigram 模型生成 5 个句子
# - 观察这些句子的特点，并进行简单解释
#
# 为了让你每次重新运行 notebook 时更容易复现结果，
# 这里会为每个模型设置固定的随机种子（random seed）。
# 这样虽然仍然是“随机生成”，但同一份代码在同一环境里通常会产生稳定结果。

NUM_SENTENCES_TO_GENERATE = 5
MAX_GENERATED_LENGTH = 20


def tokens_to_sentence(tokens):
    """把 token 列表重新拼接成便于阅读的字符串。

    由于我们前面的预处理已经移除了标点并统一转成小写，
    所以这里直接用空格把 token 拼接起来即可。
    """
    if not tokens:
        return "[empty sentence]"
    return " ".join(tokens)


generated_sentences = {}

# 依次让 3 个模型各生成 5 个句子。
for model_index, (model_name, model) in enumerate(language_models.items(), start=1):
    model_sentences = []

    for sentence_index in range(NUM_SENTENCES_TO_GENERATE):
        # 为了可复现，这里把模型编号和句子编号组合成一个固定随机种子。
        # 例如第一个模型生成第一句时使用 101，第二句用 102，以此类推。
        random_seed = model_index * 100 + sentence_index + 1

        generated_tokens = model.generate_sentence(
            max_length=MAX_GENERATED_LENGTH,
            random_seed=random_seed,
        )
        model_sentences.append(tokens_to_sentence(generated_tokens))

    generated_sentences[model_name] = model_sentences


print("Generated sentences from each language model:\n")

for model_name, sentence_list in generated_sentences.items():
    print(f"{model_name} model:")
    for sentence_number, sentence_text in enumerate(sentence_list, start=1):
        print(f"{sentence_number}. {sentence_text}")
    print()


# ==============================
# 自动打印一段模式分析
# ==============================
# 下面这部分不是“精确评价每一句内容”，
# 而是给出一个适合写作业的总体分析框架。
# 运行后你还可以根据实际生成结果，再手动补充 1~2 句更具体的观察。

print("Pattern analysis:")
print(
    "1. Unigram model usually produces the loosest sentences, because it only learns how frequent each word is and ignores word order."
)
print(
    "2. Bigram model usually generates more locally reasonable phrases, because it considers one previous word as context."
)
print(
    "3. Trigram model often gives more fluent short segments than unigram and bigram, because it uses a longer context of two previous words."
)
print(
    "4. However, n-gram models still have clear limitations: they may repeat common words, lose global coherence, or stop at awkward places."
)
print(
    "5. If the trigram results look strange, one common reason is data sparsity: many three-word contexts are still rare even after smoothing."
)


> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [ ]:
import shutil
import subprocess
from pathlib import Path

# 这一小题要做的事情比较多：
# 1. 把前面已经清洗好的训练句子写成 kenlm 可读取的纯文本语料
# 2. 用 kenlm 分别训练 bigram 和 trigram 模型
# 3. 在测试集上计算 kenlm 模型的 perplexity
# 4. 把 kenlm 的结果和我们自己实现的 bigram / trigram 结果进行比较
#
# 说明：这一题依赖外部工具和第三方库。
# 通常需要同时满足下面两个条件：
# - Python 里可以 import kenlm
# - 命令行里可以找到 lmplz 和 build_binary
#
# 如果你的环境还没准备好，运行到这里时会抛出带提示的报错，
# 这样你能马上知道缺的是哪一部分。

try:
    import kenlm
except ImportError as exc:
    raise ImportError(
        "kenlm Python package is not installed. Please install kenlm first before running this cell."
    ) from exc


def ensure_command_exists(command_name: str):
    """检查某个命令行工具是否存在。

    例如：
    - lmplz: 用来从文本语料训练 ARPA 格式语言模型
    - build_binary: 用来把 ARPA 模型转换成加载更快的二进制格式
    """
    command_path = shutil.which(command_name)
    if command_path is None:
        raise FileNotFoundError(
            f"Command '{command_name}' was not found. Please install kenlm binaries and make sure this command is in PATH."
        )
    return command_path


# 先确认训练 kenlm 所需的两个核心命令存在。
lmplz_path = ensure_command_exists("lmplz")
build_binary_path = ensure_command_exists("build_binary")

# 把本题产生的中间文件统一放到一个子目录里，避免当前目录太乱。
KENLM_DIR = Path("kenlm_artifacts")
KENLM_DIR.mkdir(exist_ok=True)

train_corpus_path = KENLM_DIR / "train_corpus.txt"
bigram_arpa_path = KENLM_DIR / "bigram.arpa"
trigram_arpa_path = KENLM_DIR / "trigram.arpa"
bigram_binary_path = KENLM_DIR / "bigram.binary"
trigram_binary_path = KENLM_DIR / "trigram.binary"


def write_sentences_for_kenlm(sentences, output_path: Path):
    """把 token 列表写成 kenlm 训练所需的一行一句的纯文本格式。

    例如一个句子 ['this', 'is', 'a', 'book']
    会被写成：
        this is a book

    注意：这里不手动写入 <s> 或 </s>，
    因为 kenlm 在评分时可以自己通过 bos/eos 参数处理句子边界。
    """
    with output_path.open("w", encoding="utf-8") as f:
        for sentence in sentences:
            if sentence:
                f.write(" ".join(sentence) + "\n")


def build_kenlm_model(order: int, corpus_path: Path, arpa_path: Path, binary_path: Path):
    """调用 kenlm 的命令行工具训练 N-Gram 模型。"""

    # 第一步：用 lmplz 从训练语料生成 ARPA 语言模型文件。
    # --discount_fallback 可以在某些数据条件下提供更稳妥的折扣回退行为。
    lmplz_command = [
        lmplz_path,
        "-o", str(order),
        "--text", str(corpus_path),
        "--arpa", str(arpa_path),
        "--discount_fallback",
    ]
    subprocess.run(lmplz_command, check=True)

    # 第二步：把 ARPA 文件转换成二进制文件。
    # 二进制格式加载更快，也更适合后续多次评分。
    build_binary_command = [
        build_binary_path,
        str(arpa_path),
        str(binary_path),
    ]
    subprocess.run(build_binary_command, check=True)


def kenlm_corpus_perplexity(model, sentences):
    """计算 kenlm 模型在一组句子上的整体 perplexity。

    这里我们自己做“整批句子的总 perplexity”计算，
    而不是简单地把每个句子的 perplexity 取平均。
    原因是：按语料总 log 概率来算，结果更标准，也更接近前面自实现模型的评估方式。

    kenlm.Model.score() 返回的是 log10 概率。
    因此这里最后要用 10 ** (...) 而不是 math.exp(... )。
    """
    total_log10_probability = 0.0
    total_predicted_tokens = 0

    for sentence in sentences:
        sentence_text = " ".join(sentence)

        # bos=True 表示句子开头使用 beginning-of-sentence 标记。
        # eos=True 表示句子结尾使用 end-of-sentence 标记。
        total_log10_probability += model.score(sentence_text, bos=True, eos=True)

        # 这里把句子结束符也计入一次预测步数。
        # 这样做和我们前面自实现模型在 perplexity 中加入 </s> 的思路保持一致。
        total_predicted_tokens += len(sentence) + 1

    return 10 ** (-total_log10_probability / total_predicted_tokens)


# 先把训练句子导出成 kenlm 训练语料文件。
write_sentences_for_kenlm(train_sentences, train_corpus_path)

# 分别训练 kenlm bigram 和 trigram。
build_kenlm_model(order=2, corpus_path=train_corpus_path, arpa_path=bigram_arpa_path, binary_path=bigram_binary_path)
build_kenlm_model(order=3, corpus_path=train_corpus_path, arpa_path=trigram_arpa_path, binary_path=trigram_binary_path)

# 加载训练好的 kenlm 模型。
kenlm_bigram_model = kenlm.Model(str(bigram_binary_path))
kenlm_trigram_model = kenlm.Model(str(trigram_binary_path))

# 计算 kenlm 模型在测试集上的困惑度。
kenlm_bigram_perplexity = kenlm_corpus_perplexity(kenlm_bigram_model, test_sentences)
kenlm_trigram_perplexity = kenlm_corpus_perplexity(kenlm_trigram_model, test_sentences)

# 同时取出我们自己实现的模型结果，便于直接对比。
# 如果你前一个代码块已经运行过，那么 perplexity_results 应该已经存在。
# 为了让这个代码块更独立稳健，如果它不存在，就现场重新算一次。
if "perplexity_results" not in globals():
    perplexity_results = {
        model_name: model.perplexity(test_sentences)
        for model_name, model in language_models.items()
    }

our_bigram_perplexity = perplexity_results["bigram"]
our_trigram_perplexity = perplexity_results["trigram"]

print("KenLM perplexity on the testing dataset:")
print(f"kenlm bigram: {kenlm_bigram_perplexity:.4f}")
print(f"kenlm trigram: {kenlm_trigram_perplexity:.4f}")

print("\nComparison with my additive-smoothing implementation:")
print(f"my bigram:    {our_bigram_perplexity:.4f}")
print(f"kenlm bigram: {kenlm_bigram_perplexity:.4f}")
print(f"my trigram:   {our_trigram_perplexity:.4f}")
print(f"kenlm trigram:{kenlm_trigram_perplexity:.4f}")


# ==============================
# 自动打印一段比较说明
# ==============================
# 一般来说，kenlm 往往会比我们自己手写的基础版 n-gram 更强，
# 原因是它的工程实现更成熟，默认的建模细节也通常更高效。
#
# 不过，最终结果还是取决于：
# - 你自己的平滑方法
# - kenlm 使用的具体估计方式
# - 数据稀疏程度
# - 词表与预处理策略是否一致

print("\nComparison analysis:")
print(
    "KenLM is a highly optimized toolkit for n-gram language modeling, so its perplexity is often lower than a simple hand-written implementation."
)
print(
    "If KenLM performs better, that usually means its modeling and implementation details handle sparse statistics more effectively."
)
print(
    "If the gap is small, it suggests that the hand-written additive-smoothing model already captures the main local word patterns reasonably well."
)
print(
    "Any remaining difference can come from smoothing strategy, sentence-boundary handling, unknown-word processing, and implementation details."
)


## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [ ]:
# 这一小题要自己实现一个 Naive Bayes 文本分类器，
# 并且明确使用 Laplace smoothing（拉普拉斯平滑，也就是“加一平滑”）。
#
# 这里我们使用的是 Multinomial Naive Bayes（多项式朴素贝叶斯），
# 它非常适合“词袋模型”场景，也就是：
# - 把一篇文档看成若干词出现了多少次
# - 不强调词在文档里的精确顺序
# - 重点利用“某些词在某些类别里更常见”这一统计规律进行分类
#
# 直观理解：
# 如果一篇文章里出现了很多政治相关词，那么它更可能属于 Politician；
# 如果出现了很多电影相关词，那么它更可能属于 Film。

from collections import defaultdict

# 为了让这个单元更稳健，这里先确保训练集和测试集里都已经有 processed_tokens。
# 如果前面 Task1 第 4 题还没运行，这里会自动补上预处理结果。
if "processed_tokens" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_tokens" not in test_records[0]:
    attach_processed_text(test_records, nlp)


def normalize_document_tokens(tokens, vocabulary):
    """把文档中的 token 规范化到训练词表空间里。

    为什么这里还要再做一次规范化？
    因为分类阶段最好和语言模型阶段保持一致：
    - 如果一个词在训练词表中，保留它
    - 如果一个词不在训练词表中，统一替换成 <UNK>

    这样可以减少“测试集出现了训练时没见过的新词”带来的问题。
    """
    return [token if token in vocabulary else UNK_TOKEN for token in tokens]


# 构造训练集和测试集的文档 token 列表以及标签列表。
# 每篇文档用一个 token 列表表示，每篇文档对应一个类别标签。
train_documents = [normalize_document_tokens(record["processed_tokens"], vocabulary) for record in train_records]
train_labels = [record["label"] for record in train_records]

test_documents = [normalize_document_tokens(record["processed_tokens"], vocabulary) for record in test_records]
test_labels = [record["label"] for record in test_records]


class MultinomialNaiveBayes:
    """一个从零实现的多项式朴素贝叶斯文本分类器。

    它会学习两类信息：
    1. 每个类别本身有多常见（先验概率 prior）
    2. 在某个类别下，每个词出现的概率（条件概率 likelihood）

    预测时，模型会比较：
    “如果这篇文档属于某个类别，那么看到这些词的概率有多大？”
    概率最大的类别，就是最终预测类别。
    """

    def __init__(self, alpha: float = 1.0):
        """初始化分类器。

        参数:
            alpha: 拉普拉斯平滑系数。
                   alpha=1.0 就是最常见的“加一平滑”。
        """
        self.alpha = alpha
        self.classes_ = []
        self.vocabulary_ = set()

        # class_document_counts[label] 记录某个类别有多少篇训练文档。
        self.class_document_counts = Counter()

        # class_token_counts[label] 记录某个类别里总共出现了多少个 token。
        # 注意这里统计的是“词总数”，不是“文档数”。
        self.class_token_counts = Counter()

        # token_counts_per_class[label][token] 记录：
        # 在某个类别下，某个词一共出现了多少次。
        self.token_counts_per_class = defaultdict(Counter)

        # 下面两个字典会在 fit() 之后计算好，供预测时直接使用。
        self.log_class_priors = {}
        self.log_token_likelihoods = defaultdict(dict)
        self.default_log_token_likelihood = {}

    def fit(self, documents, labels, vocabulary):
        """根据训练文档和标签拟合模型。"""
        self.vocabulary_ = set(vocabulary)
        self.classes_ = sorted(set(labels))
        vocabulary_size = len(self.vocabulary_)
        total_documents = len(documents)

        # 第一步：先做原始计数。
        for document_tokens, label in zip(documents, labels):
            self.class_document_counts[label] += 1

            # Counter(document_tokens) 会统计这篇文档中每个词出现了几次。
            document_counter = Counter(document_tokens)

            for token, count in document_counter.items():
                self.token_counts_per_class[label][token] += count
                self.class_token_counts[label] += count

        # 第二步：计算每个类别的先验概率 P(class)。
        # 这里取 log，是为了防止很多很小的概率连乘时数值下溢。
        for label in self.classes_:
            prior = self.class_document_counts[label] / total_documents
            self.log_class_priors[label] = math.log(prior)

        # 第三步：计算条件概率 P(token | class)。
        # 使用拉普拉斯平滑：
        # (count(token, class) + alpha) / (total_tokens_in_class + alpha * |V|)
        for label in self.classes_:
            denominator = self.class_token_counts[label] + self.alpha * vocabulary_size

            # 先把“没见过的词”的默认概率算好。
            # 这样在预测时，如果某个词在该类别下从未出现，也不会得到 0 概率。
            default_probability = self.alpha / denominator
            self.default_log_token_likelihood[label] = math.log(default_probability)

            for token in self.vocabulary_:
                token_count = self.token_counts_per_class[label][token]
                probability = (token_count + self.alpha) / denominator
                self.log_token_likelihoods[label][token] = math.log(probability)

    def predict_one(self, document_tokens):
        """预测单篇文档的类别。"""
        document_counter = Counter(document_tokens)
        class_scores = {}

        for label in self.classes_:
            # 先从该类别的先验 log 概率开始。
            score = self.log_class_priors[label]

            # 然后把文档中每个词的 log 条件概率加上去。
            # 如果一个词在文档里出现了 5 次，那么对应的 log 概率要加 5 次，
            # 所以这里写成 count * log_probability。
            for token, count in document_counter.items():
                token_log_probability = self.log_token_likelihoods[label].get(
                    token,
                    self.default_log_token_likelihood[label],
                )
                score += count * token_log_probability

            class_scores[label] = score

        # 分数最大的类别，就是模型认为最可能的类别。
        return max(class_scores, key=class_scores.get)

    def predict(self, documents):
        """批量预测多篇文档的类别。"""
        return [self.predict_one(document_tokens) for document_tokens in documents]


# 实例化并训练 Naive Bayes 分类器。
nb_classifier = MultinomialNaiveBayes(alpha=1.0)
nb_classifier.fit(train_documents, train_labels, vocabulary_list)

# 在测试集上做预测。
nb_predictions = nb_classifier.predict(test_documents)

# 先计算一个最直观的指标：accuracy（正确率）。
# 虽然题目后面要求的是 micro/macro F1，
# 但这里先打印 accuracy，有助于快速确认模型是否正常工作。
nb_accuracy = sum(pred == gold for pred, gold in zip(nb_predictions, test_labels)) / len(test_labels)

print("Naive Bayes classifier has been trained and tested on the test dataset.")
print(f"Test accuracy: {nb_accuracy:.4f}")

# 再打印少量预测样例，方便肉眼检查模型输出是否大致合理。
# 这里只展示前 10 条，避免 notebook 输出过长。
print("\nFirst 10 prediction examples:")
for index, (predicted_label, gold_label, record) in enumerate(zip(nb_predictions, test_labels, test_records), start=1):
    if index > 10:
        break
    print(f"{index}. title={record['title']} | predicted={predicted_label} | gold={gold_label}")


> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [ ]:
# 这一小题要构建 Logistic Regression（逻辑回归）文本分类器。
#
# 与 Naive Bayes 不同，逻辑回归本身不会直接“看文本字符串”。
# 它要求输入是数值特征向量。
# 所以这里我们需要先把每篇文档转换成可供模型处理的特征。
#
# 一个很经典、也很实用的做法是：
# TF-IDF + Logistic Regression
#
# 直观理解：
# - TF（Term Frequency）关注一个词在当前文档里出现得多不多
# - IDF（Inverse Document Frequency）降低那些“到处都很常见”的词的重要性
# - 两者结合后，模型更容易抓住“对区分类别真正有帮助的词”

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# 为了让这一题可以独立运行，这里先确保 processed_text 已经存在。
# processed_text 是我们在 Task1 第 4 题中保存的“清洗后重新拼接好的文档文本”。
# 例如一篇文档会变成：
# "this is a processed document ..."
#
# 对 TfidfVectorizer 来说，直接输入这种字符串形式最方便。
if "processed_text" not in train_records[0]:
    attach_processed_text(train_records, nlp)
if "processed_text" not in test_records[0]:
    attach_processed_text(test_records, nlp)

train_texts = [record["processed_text"] for record in train_records]
train_labels = [record["label"] for record in train_records]

test_texts = [record["processed_text"] for record in test_records]
test_labels = [record["label"] for record in test_records]

# 题目提示说可以把训练集切分成 train / validation 来调参。
# 这里使用 stratify=train_labels，意思是“按类别分层抽样”，
# 这样训练集和验证集中的类别分布会更接近原始数据分布。
lr_train_texts, lr_valid_texts, lr_train_labels, lr_valid_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels,
)

# 下面定义一个很小但足够合理的超参数搜索空间。
# 这样既满足“用验证集调参”的要求，也不会把 notebook 写得过于复杂。
#
# 参数含义：
# - ngram_range=(1,1)：只使用 unigram 特征
# - ngram_range=(1,2)：同时使用 unigram 和 bigram 特征
# - min_df：一个词至少在多少篇文档中出现，才保留为特征
# - C：逻辑回归中的正则化强度倒数，C 越大，正则化越弱
lr_param_grid = [
    {"ngram_range": (1, 1), "min_df": 3, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 3, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 5, "C": 1.0},
    {"ngram_range": (1, 2), "min_df": 3, "C": 2.0},
]

lr_validation_results = []
best_lr_score = -1.0
best_lr_config = None
best_lr_vectorizer = None
best_lr_model = None

# 依次尝试每组参数，并在验证集上比较 Macro-F1。
# 为什么这里优先看 Macro-F1？
# 因为这个数据集类别分布并不均衡，
# Macro-F1 会更关注“每个类别都表现得怎么样”，而不仅仅是大类。
for config in lr_param_grid:
    vectorizer = TfidfVectorizer(
        lowercase=False,
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
    )

    # 先在训练子集上拟合 TF-IDF，再把训练子集和验证子集都转换成稀疏特征矩阵。
    X_train = vectorizer.fit_transform(lr_train_texts)
    X_valid = vectorizer.transform(lr_valid_texts)

    lr_model = LogisticRegression(
        C=config["C"],
        max_iter=1000,
        solver="lbfgs",
        multi_class="auto",
    )
    lr_model.fit(X_train, lr_train_labels)

    valid_predictions = lr_model.predict(X_valid)
    valid_accuracy = accuracy_score(lr_valid_labels, valid_predictions)
    valid_macro_f1 = f1_score(lr_valid_labels, valid_predictions, average="macro")

    result_row = {
        "config": config,
        "valid_accuracy": valid_accuracy,
        "valid_macro_f1": valid_macro_f1,
    }
    lr_validation_results.append(result_row)

    if valid_macro_f1 > best_lr_score:
        best_lr_score = valid_macro_f1
        best_lr_config = config
        best_lr_vectorizer = vectorizer
        best_lr_model = lr_model


# 找到最佳参数后，按照更标准的做法：
# 在“完整训练集”上重新训练一次最终模型，
# 然后再拿这个最终模型去预测测试集。
final_lr_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=best_lr_config["ngram_range"],
    min_df=best_lr_config["min_df"],
)

X_train_full = final_lr_vectorizer.fit_transform(train_texts)
X_test = final_lr_vectorizer.transform(test_texts)

lr_classifier = LogisticRegression(
    C=best_lr_config["C"],
    max_iter=1000,
    solver="lbfgs",
    multi_class="auto",
)
lr_classifier.fit(X_train_full, train_labels)

# 在测试集上生成最终预测结果。
lr_predictions = lr_classifier.predict(X_test)
lr_test_accuracy = accuracy_score(test_labels, lr_predictions)

print("Validation results for different LR settings:")
for result_row in lr_validation_results:
    print(
        f"config={result_row['config']} | "
        f"valid_accuracy={result_row['valid_accuracy']:.4f} | "
        f"valid_macro_f1={result_row['valid_macro_f1']:.4f}"
    )

print("\nBest LR configuration:")
print(best_lr_config)
print(f"Best validation macro-F1: {best_lr_score:.4f}")

print("\nFinal Logistic Regression test result:")
print(f"Test accuracy: {lr_test_accuracy:.4f}")

# 同样打印前 10 条样例，方便和 Naive Bayes 的输出做直观比较。
print("\nFirst 10 LR prediction examples:")
for index, (predicted_label, gold_label, record) in enumerate(zip(lr_predictions, test_labels, test_records), start=1):
    if index > 10:
        break
    print(f"{index}. title={record['title']} | predicted={predicted_label} | gold={gold_label}")


> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [ ]:
# 这一小题的目标是：
# - 计算 Naive Bayes 和 Logistic Regression 在测试集上的 Micro-F1 与 Macro-F1
# - 对结果做出解释
#
# 先解释一下这两个指标的区别：
#
# 1. Micro-F1
#    会把所有类别的预测结果“放在一起”统计，
#    因此它更容易被样本量大的类别影响。
#
# 2. Macro-F1
#    会先分别计算每个类别的 F1，再对所有类别做简单平均，
#    因此它更关注“每个类别是否都被照顾到了”，
#    对类别不平衡数据集尤其重要。

from sklearn.metrics import classification_report

# 为了让这个单元能够相对独立运行，这里做两个保护判断：
# 如果前面的模型预测结果还不存在，就明确提示用户先运行前面的代码块。
if "nb_predictions" not in globals():
    raise RuntimeError("Please run the Naive Bayes cell first so that nb_predictions is available.")

if "lr_predictions" not in globals():
    raise RuntimeError("Please run the Logistic Regression cell first so that lr_predictions is available.")

nb_micro_f1 = f1_score(test_labels, nb_predictions, average="micro")
nb_macro_f1 = f1_score(test_labels, nb_predictions, average="macro")

lr_micro_f1 = f1_score(test_labels, lr_predictions, average="micro")
lr_macro_f1 = f1_score(test_labels, lr_predictions, average="macro")

print("F1-score results on the testing dataset:")
print(f"Naive Bayes    | Micro-F1: {nb_micro_f1:.4f} | Macro-F1: {nb_macro_f1:.4f}")
print(f"Logistic Reg.  | Micro-F1: {lr_micro_f1:.4f} | Macro-F1: {lr_macro_f1:.4f}")


# 分类报告会进一步展示每个类别的 precision / recall / F1。
# 这不是题目硬性要求，但非常有助于理解 Macro-F1 为什么高或低。
print("\nNaive Bayes classification report:")
print(classification_report(test_labels, nb_predictions, digits=4))

print("Logistic Regression classification report:")
print(classification_report(test_labels, lr_predictions, digits=4))


# ==============================
# 自动生成一段解释
# ==============================
print("Explanation:")

better_micro_model = "Naive Bayes" if nb_micro_f1 >= lr_micro_f1 else "Logistic Regression"
better_macro_model = "Naive Bayes" if nb_macro_f1 >= lr_macro_f1 else "Logistic Regression"

print(
    f"For Micro-F1, the better model is {better_micro_model}. This metric is heavily influenced by overall prediction correctness across all samples."
)
print(
    f"For Macro-F1, the better model is {better_macro_model}. This metric gives equal importance to every class, including minority classes."
)
print(
    "If Micro-F1 is much higher than Macro-F1, it usually means the model performs better on large classes than on small classes."
)
print(
    "That pattern is reasonable here because the dataset is imbalanced, and categories such as Politician and Film have many more documents than some smaller classes."
)
print(
    "Naive Bayes is simple and often strong for word-frequency signals, while Logistic Regression with TF-IDF usually captures more discriminative features and can perform better overall."
)


### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [1]:
# Your code goes to here




#### upload your homework: use html file format
!jupyter nbconvert assignment-01-XXX-YYY.ipynb --to html --template classic --embed-images